### **KPI'S CALCULATION & VAILDATION**

### **Modules and file import**

In [1]:
#Here importent Libraries will be imported as needed.
import pandas as pd
import numpy as np

In [2]:
#Here all the four cleaned datasets will be readed into there respective variables as dataframes.
hospital_df=pd.read_csv("C:\\powerbipro\\Health_Domain_pro_Infosys\\MedTrack_DV\\Data\\Processed_data\\Hospital_cleaned.csv")
patient_df=pd.read_csv("C:\\powerbipro\\Health_Domain_pro_Infosys\\MedTrack_DV\\Data\\Processed_data\\Patient_cleaned.csv")
department_df=pd.read_csv("C:\\powerbipro\\Health_Domain_pro_Infosys\\MedTrack_DV\\Data\\Processed_data\\Department_cleaned.csv")
resource_df=pd.read_csv("C:\\powerbipro\\Health_Domain_pro_Infosys\\MedTrack_DV\\Data\\Processed_data\\Resources_cleaned.csv")

### **KPI Calculation**

#### **1. Total Admissions**

In [3]:
#Here we are creating the total admission count measure.
Tot_admissions=hospital_df['Admission_id'].count()
print("Total Admission count : ",Tot_admissions)

Total Admission count :  353500


We calculated the total Admission count on the hospital overview dataset so we got to know that we have total 3,53,500 admission were happend in a time period of 2021 to 2025

so for this we are used the function count() :- this will count the overall overs.

#### **2. Occupancy Rate**

In [4]:
#Here we creating the Total occupancy rate % value
Tot_ocp_beds=department_df['Occupied_beds_count'].sum()
Tot_no_of_beds=department_df['Total_beds'].sum()
ocp_pct = (Tot_ocp_beds / Tot_no_of_beds) * 100
print("Total Occupancy rate % : ",round(ocp_pct,2))

Total Occupancy rate % :  68.25


Here we calculating the overall occupancy pct by before head calculating the occupied beds count and tot beds available so be simply divided them and multipled with 100 then we got the occupancy pct % value which is 68.25.

#### **3. Average Length of Stay**

In [5]:
#Here we creating the Average length of stay that a patient has been stayed in the hospital.
print("The Average length of stay (LOS) : ",round(hospital_df['Length_of_stay_days'].mean(),2))

The Average length of stay (LOS) :  4.4


Here we calculated the LOS simply by using the average function which is mean() there so . this mean will work like avg() gives us the value of LOS Avg is 4.4.

This mean's on average a patient was staying in the hospital atleast 4 days from the date of admission.

#### **4. Readmission Rate**

In [18]:
#Here we are creating the readmission rate for the patients to get to know how many admitted patients are revisting the hospital again by 
# ( tot_readmission patients / tot admissions )  * 100

filter_30_readm = hospital_df[hospital_df['Readmission_within_30_days'] == 0 ]
tot_readm_count = filter_30_readm['Readmission_flag'].sum()
tot_readm_pct = ( tot_readm_count / filter_30_readm['Admission_id'].count() ) * 100
print("Tot readmission rate % : ", round(tot_readm_pct,2))

Tot readmission rate % :  47.31


In [17]:
filter_30_readm['Admission_id'].count()

344613

Here to calculate the readmission rate first we need to verify one thing which is :

we only consider a admission is taken a readmission when the last admission date of the patient will over 30 ages so for this we already have a with in 30 days readmission column is available so first we filtered them like filtered the last 30 days readmissions.

then we calculated the readmission flag which have the 1 values respectively sum () up and them we divided with the total admission so we got the vlaue readmission ragte is 46.12

#### **5. Bed Utilization Rate**

In [7]:
#Here we are creating a measure which is tell how many beds are utilizing by the total capacity of beds available.
Available_beds= Tot_no_of_beds - Tot_ocp_beds
Bed_util_rate_pct = ( Tot_ocp_beds / Available_beds ) * 100
print("Bed_utilization rate % : ",round(Bed_util_rate_pct,2))

Bed_utilization rate % :  214.92


Here we calculating to know the overal beds utilization by calculating the occuped beds divided by the available beds so we simpley to find the available bed the substracted the tot no of beds - occuped beds then we use the tot occuped bed to divde the available beds to get the bed's utilization so the value is 214.92 that means we are using more them 100% the beds utilization.

#### **6. Department Efficiency Score**

In [8]:
#Here as per the requriment we fitch the values of the four keys from department dataset
sixth_measure=department_df.groupby('Department_name').agg({'Bed_occupancy_rate_pct' : 'mean', 
                                              'Avg_length_of_stay_days':'mean',
                                              'Readmission_rate_pct':'mean',
                                              'Equipment_downtime_hours':'mean'})

#sixth_measure['Equipment_downtime_hours'] =  round ( sixth_measure['Equipment_downtime_hours'] / 100 , 2 )
#sixth_measure

'''
#Here we are making the values of columns into normalize form by using the min-max normalization

This min max normalization will be done by two wise based on the column value if some time higer value column give us the better results 
sometimes lower value columns give us the better result so based on this we will decide wherther we have to use which type of normalization

Higher = better formula  :
score = ( (x - min(x)) / (max(x) - min(x)) ) * 100

Lower = better formula  : 
score = (( max(x) - x ) / (max(x) - min(x)) )*100

'''

#Here for the Bed occupancy score normalization we assuming this as to be " higher value = better " so based on that we are using the Higher = better formula 
lis=[]
for val in sixth_measure['Bed_occupancy_rate_pct']:
    lis.append(( ( val - min(sixth_measure['Bed_occupancy_rate_pct']) ) / ( max(sixth_measure['Bed_occupancy_rate_pct']) - min(sixth_measure['Bed_occupancy_rate_pct'])) ) * 100 )

#Here for the Avg length of stay days we assuming this as to be " lower value = better " so based on that we are using the Lower = better formula 
lis1=[]
for val in sixth_measure['Avg_length_of_stay_days']:
    lis1.append( 
        (
        ( max(sixth_measure['Avg_length_of_stay_days']) - val  ) /
        ( max(sixth_measure['Avg_length_of_stay_days']) - min(sixth_measure['Avg_length_of_stay_days']))
        ) *100
     )

#Here for the Readmission rate pct we assuming this as to be " lower value = better " so based on that we are using the Lower = better formula 
lis2=[]
for val in sixth_measure['Readmission_rate_pct']:
    lis2.append( 
        (
        ( max(sixth_measure['Readmission_rate_pct']) - val  ) /
        ( max(sixth_measure['Readmission_rate_pct']) - min(sixth_measure['Readmission_rate_pct']))
        ) *100
     )

#Here for the Equipment downtime hours we assuming this as to be " lower value = better " so based on that we are using the Lower = better formula 
lis3=[]
for val in sixth_measure['Equipment_downtime_hours']:
    lis3.append( 
        (
        ( max(sixth_measure['Equipment_downtime_hours']) - val  ) /
        ( max(sixth_measure['Equipment_downtime_hours']) - min(sixth_measure['Equipment_downtime_hours']))
        ) *100
     )

#Here we assign the list's which we are created, by directly assigning them to the columns names.
sixth_measure['Bed_occp_score'] = lis
sixth_measure['Avg LOS score'] = lis1
sixth_measure['Readm_score'] = lis2
sixth_measure['Down_time_score'] = lis3



##### **-->Weights Assignment to columns**

In [9]:
#here we are assigning the weight according to the importance of the column in the dataset.
weights_dict={
    'Bed_occp_score' : 0.30,      #Directly reflects how effectively the department is utilizing its available bed capacity. It is a major operational indicator, so it gets the highest weight.
    'Avg LOS score' : 0.25,                #Indicates how efficiently patients move through the department. Excessively long stays can reduce bed availability and increase operational burden.
    'Readm_score' : 0.25,            #Reflects potential issues with treatment effectiveness, discharge planning, or continuity of care. Since it has implications beyond simple resource utilization, it receives significant weight.
    'Down_time_score' : 0.20            #Represents lost operational capacity due to equipment/system/service downtime. Important, but its impact may be more localized than the other three indicators.
}

##### **-->We are calculating the Efficiencyscore by weights**

In [10]:
'''
Here we calculating the efficiency score by every department by taking the new df slicer part from the sixth measures
so here the formula for calculate the efficiency score 

EfficiencyScore=(OccupancyScore×0.30)+(LOSScore×0.25)+(ReadmissionScore×0.25)+(DowntimeScore×0.20)

'''
Efficiency_score=sixth_measure.iloc[:,4:]

Efficiency_score['Department_Efficiency_Score'] = round( (
    Efficiency_score[list(weights_dict.keys())]   #Here we selecting the column which we are aim to perform the multiplicate operationl , selecting keys from weights_dict converted them into the list.
    .mul(pd.Series(weights_dict))                 #Here we are converting the weights dict into the series object to match the column names of the df into row names of series.
    .sum(axis=1)                                  #then we multipling them with the respective weights
),2                                                 #Here by the sum and axis = 1 we are suming the multipled values into single by the single row wise fellowed by columns
)

Efficiency_score

,Bed_occp_score,Avg LOS score,Readm_score,Down_time_score,Department_Efficiency_Score
Department_name,,,,,
Cardiology,61.972699,19.001285,97.237975,37.035600,55.06
Emergency,100.000000,97.603131,51.571308,36.980485,74.69
Gastroenterology,24.874570,98.113811,57.835607,57.305265,57.91
General medicine,13.223610,0.000000,9.873906,45.260438,15.49
General surgery,0.000000,39.182706,52.626288,29.443586,28.84
Icu,97.633068,67.321979,35.341739,68.618427,68.68
Nephrology,70.812429,63.012994,82.133610,73.563780,72.24
Neurology,32.994535,75.376177,100.000000,72.960056,68.33
Obstetrics & gynecology,20.877581,36.498679,0.000000,99.103277,35.21


##### **-->Interpretation**

In [11]:
"""
Here we are given a textual value to clearly define how that department was performing so this is done by the numpy function.
so here we consider four case which are Highly efficient, Moderately efficient , Needs Improvements , Low efficiency.

"""

condition=[
    (Efficiency_score['Department_Efficiency_Score'] > 80 ),
    (Efficiency_score['Department_Efficiency_Score'] > 60 ) & (Efficiency_score['Department_Efficiency_Score'] <=79 ),
    (Efficiency_score['Department_Efficiency_Score'] > 40 ) & (Efficiency_score['Department_Efficiency_Score'] <=59 ),
    (Efficiency_score['Department_Efficiency_Score'] < 39 )
]

values=[
    'Highly Efficient',
    'Moderately Efficient',
    'Needs Improvement',
    'Low Efficiency'
]

Efficiency_score['Interpretation'] = np.select(condition,values)

Efficiency_score


,Bed_occp_score,Avg LOS score,Readm_score,Down_time_score,Department_Efficiency_Score,Interpretation
Department_name,,,,,,
Cardiology,61.972699,19.001285,97.237975,37.035600,55.06,Needs Improvement
Emergency,100.000000,97.603131,51.571308,36.980485,74.69,Moderately Efficient
Gastroenterology,24.874570,98.113811,57.835607,57.305265,57.91,Needs Improvement
General medicine,13.223610,0.000000,9.873906,45.260438,15.49,Low Efficiency
General surgery,0.000000,39.182706,52.626288,29.443586,28.84,Low Efficiency
Icu,97.633068,67.321979,35.341739,68.618427,68.68,Moderately Efficient
Nephrology,70.812429,63.012994,82.133610,73.563780,72.24,Moderately Efficient
Neurology,32.994535,75.376177,100.000000,72.960056,68.33,Moderately Efficient
Obstetrics & gynecology,20.877581,36.498679,0.000000,99.103277,35.21,Low Efficiency


In [12]:
chek=department_df.groupby('Department_name').agg({
    'Equipment_downtime_hours' : 'sum',
    'Department_name' : 'count'
})

#chek['Tot_occup_pct']= ( chek['Occupied_beds_count'] / chek['Total_beds'] ) * 100
chek

,Equipment_downtime_hours,Department_name
Department_name,,
Cardiology,9075.27,18402
Emergency,9071.23,18393
Gastroenterology,8924.27,18393
General medicine,9019.69,18410
General surgery,9131.68,18405
Icu,8842.95,18394
Nephrology,8815.33,18411
Neurology,8824.49,18421
Obstetrics & gynecology,8628.14,18406


In [13]:
pd.set_option('display.max_columns',46)
department_df.head()

,Dept_analytics_id,Date,Year,Month,Quarter,Day_of_week,Is_weekend,Hospital_id,Hospital_name,Department_id,Department_name,Department_type,Floor_number,Is_24x7,Established_year,Department_head_name,Total_beds,Occupied_beds_count,Bed_occupancy_rate_pct,Target_occupancy_rate_pct,Patients_admitted_count,Patients_discharged_count,Readmission_count,Readmission_rate_pct,Target_readmission_rate_pct,Mortality_count,Mortality_rate_pct,Avg_length_of_stay_days,Target_avg_los_days,Avg_treatment_time_hours,Transfer_events_count,Bed_turnaround_time_hours,Nurses_on_duty,Doctors_on_duty,Base_nurses_per_shift,Base_doctors_per_shift,Staff_to_patient_ratio,Equipment_downtime_hours,Avg_patient_age,Avg_satisfaction_score,Total_billing_usd,Avg_cost_per_patient_usd,Monthly_budget_allocated_usd,Department_efficiency_score,Record_source_system,Last_updated_timestamp
0,DA00116235,2024-04-11,2024,4,2,Thursday,False,H005,Metro city hospital,D04,Oncology,Non-surgical,4,False,2001,Dr. Carlos Ivanov,46,36,78.26,86.68,2,4,2,100.00,11.57,0,0.0,4,3.87,58.76,4,0.78,5,3,7,4,0.14,0.14,49,2.0,4131.26,2065.63,175431.27,0.0,Analytics-etl,2024-04-12
1,DA00092491,2024-04-05,2024,4,2,Friday,False,H004,Sunrise health institute,D06,General surgery,Surgical,4,False,2007,Dr. Elena Brown,30,22,73.33,84.65,1,1,0,0.00,9.41,0,0.0,5,5.69,49.20,4,3.23,5,5,3,5,0.23,0.16,19,5.0,6515.81,6515.81,94570.59,60.7,Analytics-etl,2024-04-06
2,DA00097464,2022-11-17,2022,11,4,Thursday,False,H004,Sunrise health institute,D09,Nephrology,Non-surgical,4,False,2000,Dr. Ivan Rossi,51,35,68.63,82.74,3,1,2,66.67,7.68,0,0.0,6,6.17,79.02,4,1.47,3,6,3,5,0.09,0.49,38,6.0,30686.79,10228.93,231074.98,0.0,Analytics-etl,2022-11-19
3,DA00225118,2022-06-04,2022,6,2,Saturday,True,H009,Pinecrest general hospital,D04,Oncology,Non-surgical,3,False,2015,Dr. Liam Ivanov,29,17,58.62,81.22,1,2,0,0.00,12.56,0,0.0,9,5.67,67.68,3,0.20,5,4,7,4,0.29,0.36,8,3.0,18952.22,18952.22,378377.47,42.3,Analytics-etl,2022-06-08
4,DA00054680,2025-09-22,2025,9,3,Monday,False,H002,Lakeview medical center,D15,Urology,Surgical,2,True,2005,Dr. Nadia Chen,42,28,66.67,71.10,2,1,2,100.00,8.72,0,0.0,7,5.80,119.76,6,1.10,4,4,6,5,0.14,0.14,37,1.0,22770.44,11385.22,221023.35,0.0,Manual aggregation,2025-09-24


###

In [14]:
department_df['Equipment_downtime_hours'].unique()

array([ 0.14,  0.16,  0.49, ..., 38.21, 29.22, 22.24])